In [1]:
import pandas as pd
import dash
from dash import dcc, html, Input, Output
import plotly.express as px


# Load CGAM stability dataset
# Make sure this matches your exported CSV
stability_df = pd.read_csv(
    r"..\..\SavedOutcomesAim2\Overground_EMG_Kinematics\2_FromPython_CGAM\CGAM_Stability_1000Runs.csv"
)


# Print feature list for a given run
def get_features_for_run(run_number):
    """
    Returns the list of features used in that run.
    """
    subset = stability_df[stability_df["Run"] == run_number]
    if subset.empty:
        print(f"No run found with Run={run_number}")
        return None
    
    feat_str = subset["SelectedFeatures"].iloc[0]
    features = feat_str.split(",")
    print(f"\nRun {run_number} features:")
    for f in features:
        print("  -", f)
    return features



# Create two filtered dataframes: FV and SSV
df_FV = stability_df[stability_df["Speed"] == "FV"]
df_SSV = stability_df[stability_df["Speed"] == "SSV"]


# Create Dash App
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("CGAM Stability Visualization (1000 Runs)", style={"textAlign": "center"}),

    html.Div([
        html.Div([
            html.H3("Fast Velocity (FV)"),
            dcc.Graph(
                id='scatter-fv',
                figure=px.scatter(
                    df_FV,
                    x="Run",
                    y="CGAM",
                    color="Intervention",
                    hover_data=["Subject", "Trial", "Cycle", "SelectedFeatures"],
                    title="CGAM Values Across 1000 Runs (FV)"
                )
            )
        ], style={"width": "48%", "display": "inline-block"}),

        html.Div([
            html.H3("Self-Selected Velocity (SSV)"),
            dcc.Graph(
                id='scatter-ssv',
                figure=px.scatter(
                    df_SSV,
                    x="Run",
                    y="CGAM",
                    color="Intervention",
                    hover_data=["Subject", "Trial", "Cycle", "SelectedFeatures"],
                    title="CGAM Values Across 1000 Runs (SSV)"
                )
            )
        ], style={"width": "48%", "display": "inline-block", "float": "right"}),
    ]),

    html.Hr(),

    html.H3("Clicked Point Information"),
    html.Div(id='clicked-info', style={"whiteSpace": "pre-line", "fontSize": 16, "marginTop": "10px"}),

])



# Display info when clicking FV or SSV plot
@app.callback(
    Output('clicked-info', 'children'),
    Input('scatter-fv', 'clickData'),
    Input('scatter-ssv', 'clickData')
)
def display_click(clickFV, clickSSV):
    ctx = dash.callback_context

    if not ctx.triggered:
        return "Click a point in either scatter plot."

    trigger = ctx.triggered[0]['prop_id'].split('.')[0]

    clickData = clickFV if trigger == 'scatter-fv' else clickSSV

    if clickData is None:
        return "Click a point in either scatter plot."

    point = clickData['points'][0]
    run_number = point['x']
    cgam_val = point['y']
    intervention = point['marker.color']

    # Extract features
    df_match = stability_df[stability_df["Run"] == run_number]
    feat_str = df_match["SelectedFeatures"].iloc[0]

    return (
        f"Run: {run_number}\n"
        f"Intervention: {intervention}\n"
        f"CGAM Value: {cgam_val:.4f}\n\n"
        f"Features Used:\n{feat_str.replace(',', ', ')}"
    )



# RUN SERVER
if __name__ == "__main__":
    print("\nGUI Running at: http://127.0.0.1:8050\n")
    print("Use get_features_for_run(run_number) in Python to print a feature list.")
    app.run_server(debug=True)


FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\SavedOutcomesAim2\\Overground_EMG_Kinematics\\2_FromPython_CGAM\\CGAM_Stability_1000Runs.csv'

In [ ]:
# Run the Dash GUI (this will open in a browser)
app.run_server(mode='external', debug=True)
